# S-ATTP: cluster-stratified train/test partitioning and its validation limits

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jabualani/ML-Reliabilty/blob/main/docs/colab/attp-method.ipynb)

Companion notebook for the ReliableML paper `docs/latex/attp-method.tex`. S-ATTP lives in the companion package ATTP (`i-attp`), which ReliableML recommends for partitioning. This notebook uses the real `attp.s_attp_clf` API and then reproduces its LDA -> K-means -> strata recipe to expose the selection and leakage issues the paper discusses.

All numbers shown below are computed by the cells themselves (fixed random seeds); nothing is hard-coded. Run the cells top to bottom.

## 0. Setup (parameters)

**Parameter cell - choose ONE installation source.** The default installs the released package from PyPI. To try the unreleased development version, comment out option A and uncomment option B. If neither works (for example a restricted network), run the fallback cell below it and upload a zip of the repository.

In [ ]:
#@title PARAMETERS: install ReliableML
# ---- Option A (default): released version from PyPI ----
!pip install -q ReliableML
# ---- Option B: unreleased version from GitHub (uncomment; comment option A) ----
# !pip install -q "git+https://github.com/Jabualani/ML-Reliabilty.git"
# Plots need matplotlib (preinstalled on Colab); locally: pip install matplotlib
# ATTP is a separate package (the S-ATTP classifier partitioner used in this notebook)
!pip install -q i-attp==0.2.3

In [ ]:
#@title Fallback: upload a repository zip (only runs if ml_reliability is not importable)
import importlib.util, pathlib, sys, zipfile

if importlib.util.find_spec("ml_reliability") is None:
    from google.colab import files  # Colab only

    # GitHub: Code > Download ZIP, then pick the file in the dialog
    uploaded = files.upload()
    for archive in uploaded:
        zipfile.ZipFile(archive).extractall("repo_zip")
    package_init = next(pathlib.Path("repo_zip").rglob("ml_reliability/__init__.py"))
    sys.path.insert(0, str(package_init.parent.parent))
    # the repo's dependencies (numpy, pandas, scikit-learn, scipy, statsmodels) are
    # preinstalled on Colab; otherwise: !pip install -q numpy pandas scikit-learn scipy statsmodels

import ml_reliability
print("ml_reliability loaded from:", pathlib.Path(ml_reliability.__file__).parent)

## 1. Method

Let $X\in\mathbb R^{n\times p}$ be a finite numeric matrix and $y_i\in\{1,\dots,C\}$.

**Supervised projection (LDA).** With class means $\bar x_c$, global mean $\bar x$ and class sizes $n_c$,
$$S_W=\sum_{c}\sum_{i:y_i=c}(x_i-\bar x_c)(x_i-\bar x_c)^\top,\qquad S_B=\sum_c n_c(\bar x_c-\bar x)(\bar x_c-\bar x)^\top,\qquad J(w)=\frac{w^\top S_Bw}{w^\top S_Ww}.$$
At most $\min(p,C-1)$ discriminant components exist; components are added until the cumulative explained variance reaches a threshold (default $0.85$), giving $Z=XW$.

**Clusters and joint strata.** For each candidate $k$, K-means minimizes $J_k=\sum_{j=1}^k\sum_{i:z_i\in C_j}\lVert z_i-\mu_j\rVert_2^2$. The stratum of row $i$ is $S_i=(y_i,c_i)$. Rows in singleton strata cannot appear in both partitions and are always assigned to **training**.

**Candidate selection.** For each train fraction $\alpha$ and cluster count $k$ a classifier $h_{\alpha,k}$ is fitted on the training part and scored on the candidate test part:
$$(\alpha^\star,k^\star)=\arg\max_{(\alpha,k)\in\mathcal A}\widehat Q_{\mathrm{test}}(h_{\alpha,k}).$$

**Why the returned test score is not a final estimate.** LDA and K-means see *all* rows supplied to the call before any split exists, and the maximum over candidates is itself selected on the test part (selection optimism). A defensible protocol holds out an untouched outer set *first* and runs everything else, including `s_attp_clf`, on the remaining data.

## 2. Run S-ATTP on a 3-class dataset and inspect the result

Scikit-learn's wine dataset (13 numeric features, 3 classes) is a bundled data source here; it gives 2 LDA components, which can be plotted.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import attp
from sklearn.cluster import KMeans
from sklearn.datasets import load_wine, load_breast_cancer, make_classification
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

wine = load_wine(as_frame=True)
Xw, yw = wine.data, wine.target
part = attp.s_attp_clf(Xw, yw, splits=(0.7, 0.8), min_clusters=2, max_clusters=8, random_state=0)

print("best train fraction :", part.best_split)
print("best #clusters      :", part.best_n_clusters)
print("LDA components      :", part.n_lda_components, "| explained variance ratio:", np.round(part.explained_variance_ratio, 3),
      "| total:", round(part.total_explained_variance, 3))
print("train/test rows     :", len(part.X_train), "/", len(part.X_test))
print("internal scores     : train=%.3f test=%.3f" % (part.train_score, part.test_score))
part.metrics.head(8)

The `metrics` table lists every feasible candidate $(\alpha,k)$; the winner is simply the row with the largest test score.

In [ ]:
metrics = part.metrics
print("candidates evaluated:", len(metrics))
print("max candidate test score : %.3f" % metrics["test_score"].max())
print("mean candidate test score: %.3f" % metrics["test_score"].mean())
print("spread (max - min)       : %.3f" % (metrics["test_score"].max() - metrics["test_score"].min()))

## 3. Reproduce the recipe: LDA projection, clusters and strata

The same scikit-learn calls the package uses (LDA with the selected number of components, then K-means with the selected $k$) are re-run to draw the projection and count singleton strata.

In [ ]:
lda = LinearDiscriminantAnalysis(n_components=part.n_lda_components).fit(Xw, yw)
Z = lda.transform(Xw)
clusters = KMeans(n_clusters=part.best_n_clusters, random_state=0, n_init="auto").fit_predict(Z)
strata = pd.crosstab(pd.Series(yw.to_numpy(), name="class"), pd.Series(clusters, name="cluster"))
print(strata)
sizes = pd.Series(list(zip(yw.to_numpy(), clusters))).value_counts()
singletons = int((sizes == 1).sum())
print("singleton strata (forced into training):", singletons)

in_train = Xw.index.isin(part.train_indices)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
zy = Z[:, 1] if Z.shape[1] > 1 else np.zeros(len(Z))
axes[0].scatter(Z[:, 0], zy, c=yw, cmap="viridis", s=18); axes[0].set_title("LDA projection coloured by class")
axes[1].scatter(Z[:, 0], zy, c=clusters, cmap="tab10", s=18)
axes[1].scatter(Z[~in_train, 0], zy[~in_train], facecolors="none", edgecolors="k", s=60, label="partition test")
axes[1].set_title(f"K-means clusters (k={part.best_n_clusters}); circles = test rows"); axes[1].legend()
for ax in axes: ax.set_xlabel("LD1"); ax.set_ylabel("LD2")
plt.tight_layout(); plt.show()

print("class proportions  all/train/test:",
      np.round(yw.value_counts(normalize=True).sort_index().to_numpy(), 3),
      np.round(part.y_train.value_counts(normalize=True).sort_index().to_numpy(), 3),
      np.round(part.y_test.value_counts(normalize=True).sort_index().to_numpy(), 3))

## 4. Outer-evaluation protocol on a bundled binary dataset

Breast cancer: hold out an **untouched outer test set first**, run `s_attp_clf` on the remaining development data only, fit the final model on the ATTP training rows, and evaluate once on the outer set. Repeating over several outer splits compares ATTP's internal (selected) test score with the honest outer score.

In [ ]:
bc = load_breast_cancer(as_frame=True)
Xb, yb = bc.data, bc.target
rows = []
for seed in range(8):
    X_dev, X_outer, y_dev, y_outer = train_test_split(Xb, yb, test_size=0.3, stratify=yb, random_state=seed)
    result = attp.s_attp_clf(X_dev, y_dev, splits=(0.7, 0.8), min_clusters=2, max_clusters=6, random_state=seed)
    final = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(result.X_train, result.y_train)
    rows.append({"outer seed": seed, "k*": result.best_n_clusters,
                 "ATTP internal test score": result.test_score,
                 "outer test accuracy": final.score(X_outer, y_outer)})
outer = pd.DataFrame(rows)
outer["internal - outer"] = outer["ATTP internal test score"] - outer["outer test accuracy"]
print(outer.round(3).to_string(index=False))
print("mean internal - outer difference: %.3f (sd %.3f); note the two scores use different classifiers" %
      (outer["internal - outer"].mean(), outer["internal - outer"].std()))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(outer["outer seed"], outer["ATTP internal test score"], "o-", label="ATTP internal test score")
ax.plot(outer["outer seed"], outer["outer test accuracy"], "s-", label="untouched outer test accuracy")
ax.set_xlabel("outer split seed"); ax.set_ylabel("accuracy"); ax.legend(); plt.tight_layout(); plt.show()

## 5. Synthetic data: a rare class creates singleton strata

With a small minority class and many clusters, some class-cluster cells contain one row; those rows are forced into training, so the realized split can depart from the nominal fraction.

In [ ]:
Xs, ys = make_classification(n_samples=300, n_features=6, n_informative=4, n_redundant=0,
                             weights=[0.93, 0.07], class_sep=1.0, random_state=3)
Xs = pd.DataFrame(Xs, columns=[f"f{i}" for i in range(6)]); ys = pd.Series(ys)
res = attp.s_attp_clf(Xs, ys, splits=(0.8,), min_clusters=6, max_clusters=10, random_state=0)

lda_s = LinearDiscriminantAnalysis(n_components=res.n_lda_components).fit(Xs, ys)
cl = KMeans(n_clusters=res.best_n_clusters, random_state=0, n_init="auto").fit_predict(lda_s.transform(Xs))
cell_sizes = pd.Series(list(zip(ys.to_numpy(), cl))).value_counts()
print("minority rows:", int((ys == 1).sum()), "| k* =", res.best_n_clusters, "| singleton strata:", int((cell_sizes == 1).sum()))
print("nominal train fraction: %.2f | realized train fraction: %.3f" % (res.best_split, len(res.X_train) / len(Xs)))
print("minority rows in test:", int((res.y_test == 1).sum()), "of", int((ys == 1).sum()))

## Interpretation and limitations

**How to read the output.** `SplitResult` exposes the chosen train fraction, cluster count, LDA components, explained variance, internal scores and the full candidate table. The internal test score is a *tuning criterion*; the outer-protocol table shows how it compares with scores on data that took no part in the search. The differences also depend on the classifier used inside ATTP versus the final model.

**Limitations.** LDA is supervised and linear, K-means prefers spherical Euclidean clusters and is scale-sensitive; inputs must be finite numeric (no imputation, encoding or scaling is done); singleton strata always go to training; a score-based choice can favor splits that are easy for the chosen classifier rather than representative of a target population. S-ATTP neither detects nor guarantees an out-of-distribution validation set, and temporal or grouped deployments need temporal/group outer splits. Only the S-ATTP classifier partitioner is covered; the wider ATTP taxonomy is out of scope.

These notebooks are methodological demonstrations on synthetic data and small bundled scikit-learn datasets. They do not show that any method improves performance on your data; that requires a prespecified evaluation on independent data.